In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')
print("Imports successful")

Imports successful


In [2]:
app_train = pd.read_csv('../data/application_train.csv')
app_test = pd.read_csv('../data/application_test.csv')
bureau = pd.read_csv('../data/bureau.csv')
prev_app = pd.read_csv('../data/previous_application.csv')

print("Data loaded")
print("app_train shape:", app_train.shape)

Data loaded
app_train shape: (307511, 122)


In [3]:
total_rows = len(app_train)

missing_percent = app_train.isnull().sum() / total_rows * 100

cols_to_drop = missing_percent[missing_percent > 50].index.tolist()

app_train = app_train.drop(columns=cols_to_drop)
app_test = app_test.drop(columns=cols_to_drop)

print("Columns dropped:", len(cols_to_drop))
print("Remaining columns:", app_train.shape[1])

Columns dropped: 41
Remaining columns: 81


In [4]:
app_train['AGE'] = app_train['DAYS_BIRTH'] / -365
app_test['AGE'] = app_test['DAYS_BIRTH'] / -365

app_train = app_train.drop(columns=['DAYS_BIRTH'])
app_test = app_test.drop(columns=['DAYS_BIRTH'])

print("AGE column created ✓")
print("Sample ages:", app_train['AGE'].head().values)

AGE column created ✓
Sample ages: [25.92054795 45.93150685 52.18082192 52.06849315 54.60821918]


In [5]:
app_train['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)
app_test['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)

app_train['YEARS_EMPLOYED'] = app_train['DAYS_EMPLOYED'] / -365
app_test['YEARS_EMPLOYED'] = app_test['DAYS_EMPLOYED'] / -365

app_train = app_train.drop(columns=['DAYS_EMPLOYED'])
app_test = app_test.drop(columns=['DAYS_EMPLOYED'])

print("YEARS_EMPLOYED column created ✓")

YEARS_EMPLOYED column created ✓


In [6]:
app_train['LOAN_TO_INCOME'] = app_train['AMT_CREDIT'] / app_train['AMT_INCOME_TOTAL']
app_test['LOAN_TO_INCOME'] = app_test['AMT_CREDIT'] / app_test['AMT_INCOME_TOTAL']

print("LOAN_TO_INCOME created ✓")
print("Average loan-to-income:", round(app_train['LOAN_TO_INCOME'].mean(), 2))

LOAN_TO_INCOME created ✓
Average loan-to-income: 3.96


In [7]:
app_train['ANNUITY_TO_INCOME'] = app_train['AMT_ANNUITY'] / app_train['AMT_INCOME_TOTAL']
app_test['ANNUITY_TO_INCOME'] = app_test['AMT_ANNUITY'] / app_test['AMT_INCOME_TOTAL']

print("ANNUITY_TO_INCOME created ✓")

ANNUITY_TO_INCOME created ✓


In [8]:
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    BUREAU_LOAN_COUNT=('SK_ID_BUREAU', 'count'),
    BUREAU_CREDIT_SUM=('AMT_CREDIT_SUM', 'sum'),
    BUREAU_CREDIT_MEAN=('AMT_CREDIT_SUM', 'mean'),
    BUREAU_OVERDUE_MAX=('AMT_CREDIT_SUM_OVERDUE', 'max')
)

bureau_agg = bureau_agg.reset_index()

print("Bureau aggregation done ✓")
bureau_agg.head()

Bureau aggregation done ✓


,SK_ID_CURR,BUREAU_LOAN_COUNT,BUREAU_CREDIT_SUM,BUREAU_CREDIT_MEAN,BUREAU_OVERDUE_MAX
0,100001,7,1453365.000,207623.571429,0.0
1,100002,8,865055.565,108131.945625,0.0
2,100003,4,1017400.500,254350.125000,0.0
3,100004,2,189037.800,94518.900000,0.0
4,100005,3,657126.000,219042.000000,0.0


In [9]:
prev_agg = prev_app.groupby('SK_ID_CURR').agg(
    PREV_APP_COUNT = ('SK_ID_PREV', 'count'),
    PREV_AMT_CREDIT_MEAN = ('AMT_CREDIT', 'mean'),
    PREV_AMT_CREDIT_MAX = ('AMT_CREDIT', 'max')
)

prev_agg = prev_agg.reset_index()

print("Previous application aggregation done ✓")
print("Shape:", prev_agg.shape)
prev_agg.head()

Previous application aggregation done ✓
Shape: (338857, 4)


,SK_ID_CURR,PREV_APP_COUNT,PREV_AMT_CREDIT_MEAN,PREV_AMT_CREDIT_MAX
0,100001,1,23787.00,23787.0
1,100002,1,179055.00,179055.0
2,100003,3,484191.00,1035882.0
3,100004,1,20106.00,20106.0
4,100005,2,20076.75,40153.5


In [10]:
app_train = app_train.merge(bureau_agg, on='SK_ID_CURR', how='left')
app_train = app_train.merge(prev_agg, on='SK_ID_CURR', how='left')

app_test = app_test.merge(bureau_agg, on='SK_ID_CURR', how='left')
app_test = app_test.merge(prev_agg, on='SK_ID_CURR', how='left')

print("Merging done ✓")
print("app_train shape after merge:", app_train.shape)

Merging done ✓
app_train shape after merge: (307511, 90)


In [11]:
num_cols = app_train.select_dtypes(include='number').columns.tolist()
num_cols.remove('TARGET')

for col in num_cols:
    median_val = app_train[col].median()
    app_train[col] = app_train[col].fillna(median_val)
    app_test[col] = app_test[col].fillna(median_val)

print("Done ✓")

Done ✓


In [12]:
cat_cols = app_train.select_dtypes(include='object').columns.tolist()

for col in cat_cols:
    mode_val = app_train[col].mode()[0]
    app_train[col] = app_train[col].fillna(mode_val)
    app_test[col] = app_test[col].fillna(mode_val)

print("Categorical nulls filled with mode ✓")
print("Remaining nulls in train:", app_train[cat_cols].isnull().sum().sum())

Categorical nulls filled with mode ✓
Remaining nulls in train: 0


In [13]:
app_train = pd.get_dummies(app_train, columns=cat_cols)
app_test = pd.get_dummies(app_test, columns=cat_cols)

app_train, app_test = app_train.align(app_test, join='left', axis=1, fill_value=0)

print("Train shape after encoding:", app_train.shape)
print("Test shape after encoding: ", app_test.shape)

Train shape after encoding: (307511, 203)
Test shape after encoding:  (48744, 203)


In [14]:
X = app_train.drop(columns=['TARGET', 'SK_ID_CURR'])
y = app_train['TARGET']

X_test_final = app_test.drop(columns=['TARGET', 'SK_ID_CURR'], errors='ignore')

print("Features shape:", X.shape)
print("Target shape:  ", y.shape)
print("Default rate:  ", round(y.mean() * 100, 1), "%")

Features shape: (307511, 201)
Target shape:   (307511,)
Default rate:   8.1 %


In [15]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:  ", X_train.shape)
print("Validation set:", X_val.shape)
print("\nDefault rate in train:", round(y_train.mean() * 100, 1), "%")
print("Default rate in val:  ", round(y_val.mean() * 100, 1), "%")

Training set:   (246008, 201)
Validation set: (61503, 201)

Default rate in train: 8.1 %
Default rate in val:   8.1 %


In [16]:
import os

os.makedirs('../outputs', exist_ok=True)

X_train.to_csv('../outputs/X_train.csv', index=False)
X_val.to_csv('../outputs/X_val.csv', index=False)
y_train.to_csv('../outputs/y_train.csv', index=False)
y_val.to_csv('../outputs/y_val.csv', index=False)

print("All files saved")


All files saved
